In [13]:
%pip install -U langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers datasets tqdm

  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.1
    Uninstalling tqdm-4.67.1:
      Successfully uninstalled tqdm-4.67.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.4.1
    Uninstalling datasets-4.4.1:
      Successfully uninstalled datasets-4.4.1
  Attempting uninstall: sentence-transformers━━━━━━━━━━━━━━━━━━━━━ 1/3 [datasets]
    Found existing installation: sentence-transformers 5.1.2━━ 1/3 [datasets]
    Uninstalling sentence-transformers-5.1.2:━━━━━━━━━━━━━━━━━ 1/3 [datasets]
      Successfully uninstalled sentence-transformers-5.1.2━━━━ 1/3 [datasets]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [sentence-transformers]ence-transformers]
Note: you may need to restart the kernel to use updated packages.


In [17]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large-instruct",
    model_kwargs={"device": "mps"} 
)

In [24]:
from langchain_community.vectorstores import FAISS

# Load index
vectorstore = FAISS.load_local(
    "/Users/novay/Applications/Project/Enterwind/lexindo/python/lexindo-llm/04_rag_corpus/faiss_index_lexindo", 
    embeddings,
    allow_dangerous_deserialization=True  # wajib di versi LangChain baru, karena pickle-based
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})  # top-3 chunks
print("Retriever FAISS berhasil di-load dari index lokal!")

Retriever FAISS berhasil di-load dari index lokal!


In [25]:
test_question = "Apa tujuan Manajemen Sumber Daya Manusia menurut Pasal 20?"
retrieved = retriever.invoke(test_question)
print("Retrieved contexts:")
for doc in retrieved:
    print(doc.page_content[:200] + "...")  # preview

Retrieved contexts:
Pasal 20 Manajemen Sumber Daya Manusia sebagaimana dimaksud dalam Pasal 15 ayat (2) huruf e dilakukan untuk menjamin ketersediaan dan meningkatkan kompetensi pegawai ASN dalam penyelenggaraan SPBE mel...
Pasal 5 Maksud dan tujuan Penyesuaian Sistem Kerja yaitu: a. mewujudkan proses kerja yang efektif dan efisien; b. memastikan pencapaian tujuan, strategi, dan kinerja organisasi; c. mengoptimalkan pema...
Pasal 17. (1) Sub Bagian Kepegawaian dan Pengembangan SDM dipimpin oleh Kepala Sub Bagian yang bertanggung jawab kepada Bagian Umum, SDM, dan Keuangan. (2) Bertugas memimpin, mengelola administrasi ke...


In [26]:
from datasets import load_dataset
from langchain_community.llms import Ollama

# Load dataset evaluasi
eval_dataset = load_dataset("json", data_files="/Users/novay/Applications/Project/Enterwind/lexindo/python/lexindo-llm/03_finetune_dataset/data/split/ragas_eval.jsonl")["train"]
eval_dataset[0]

{'question': 'Apa tujuan Manajemen Sumber Daya Manusia menurut Pasal 20?',
 'ground_truth': 'Berdasarkan Peraturan Bupati Nomor 45 Tahun 2022 tentang Sistem Pemerintahan Berbasis Elektronik Dalam Penyelenggaraan Pemerintahan Daerah, Pasal 20 menyatakan bahwa:\n\nPasal 20 Manajemen Sumber Daya Manusia sebagaimana dimaksud dalam Pasal 15 ayat (2) huruf e dilakukan untuk menjamin ketersediaan dan meningkatkan kompetensi pegawai ASN dalam penyelenggaraan SPBE melalui pendidikan dan pelatihan.\n\n(Sumber: Peraturan Bupati Nomor 45 Tahun 2022, Pasal 20)',
 'answer': 'Berdasarkan Peraturan Bupati Nomor 45 Tahun 2022 tentang Sistem Pemerintahan Berbasis Elektronik Dalam Penyelenggaraan Pemerintahan Daerah, Pasal 20 menyatakan bahwa:\n\nPasal 20 Manajemen Sumber Daya Manusia sebagaimana dimaksud dalam Pasal 15 ayat (2) huruf e dilakukan untuk menjamin ketersediaan dan meningkatkan kompetensi pegawai ASN dalam penyelenggaraan SPBE melalui pendidikan dan pelatihan.',
 'contexts': ['(Sumber:Peratu

In [27]:
# LLM via Ollama
llm_lexindo = Ollama(model="lexindo-custom", temperature=0.3, num_ctx=1024)

In [29]:
input_file = '/Users/novay/Applications/Project/Enterwind/lexindo/python/lexindo-llm/05_evaluation_set/ragas_eval_beautified.jsonl'  # sesuaikan
output_file = '/Users/novay/Applications/Project/Enterwind/lexindo/python/lexindo-llm/05_evaluation_set/ragas_eval_finetuned_rag_batch.jsonl'  # nama per batch

# Load data (sama seperti script sebelumnya)
with open(input_file, 'r', encoding='utf-8') as f:
    content = f.read()
    import re
    blocks = re.findall(r'\{.*?\}(?=\n\{|\n$|$)', content, re.DOTALL)
    data_list = [json.loads(block) for block in blocks]

print(f"Total data: {len(data_list)}")

Total data: 1119


In [30]:
start_idx = 0 
batch_size = 1118
end_idx = min(start_idx + batch_size, len(data_list))

updated_partial = []
for i in tqdm(range(start_idx, end_idx)):
    data = data_list[i]
    question = data['question']

    try:
        retrieved = retriever.invoke(question)
        context_str = "\n\n".join([doc.page_content for doc in retrieved])

        prompt = f"""Gunakan HANYA informasi dari konteks berikut. Jawab secara akurat, lengkap, to the point, dan cantumkan sumber jika relevan.

Konteks:
{context_str}

Pertanyaan: {question}
Jawaban:"""

        answer = llm_lexindo.invoke(prompt).strip()
        data['answer_finetuned_rag'] = answer
        data['contexts'] = [doc.page_content for doc in retrieved]  # simpan fresh
    except Exception as e:
        print(f"Error index {i}: {e}")
        data['answer_finetuned_rag'] = "ERROR_RAG"
        data['contexts'] = []

    updated_partial.append(data)

# Simpan batch ini
with open(output_file, 'w', encoding='utf-8') as f:
    for entry in updated_partial:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f"Batch {start_idx}-{end_idx-1} selesai. File: {output_file}")
print(f"Lanjut batch berikutnya: ubah start_idx ke {end_idx}")

100%|██████████| 1118/1118 [2:28:25<00:00,  7.97s/it]  

Batch 0-1117 selesai. File: /Users/novay/Applications/Project/Enterwind/lexindo/python/lexindo-llm/05_evaluation_set/ragas_eval_finetuned_rag_batch.jsonl
Lanjut batch berikutnya: ubah start_idx ke 1118
